# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kobeyvines/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Five-feature frame for Lane 1 (Ranking Signal Analysis):**

| Feature | What it captures |
|---|---|
| `impressions_90d` | demand/visibility — is anyone seeing this page? |
| `avg_position` | current ranking signal |
| `ctr` | click behavior given that ranking |
| `days_since_last_update` | staleness/freshness |
| `word_count` | content depth |

Kept deliberately small: five observable signals, each tied to a distinct question a content team actually asks (is it seen, where does it rank, does it get clicked, is it old, is it thin). This is the same frame w04's baseline rule and signal checks build on (CTR-by-position-tier, staleness-by-impressions), so the feature definitions here need to be the ones that carry forward, not a separate set.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RELATIVE_PATH = Path("data/raw/content_refresh_anonymized.csv")
DATA_PATH = next(
    (base / DATA_RELATIVE_PATH for base in [Path.cwd(), *Path.cwd().parents]
     if (base / DATA_RELATIVE_PATH).is_file()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_RELATIVE_PATH} from the current directory or its parents. "
        f"Current directory: {Path.cwd()}"
    )
raw = pd.read_csv(DATA_PATH)

# Same inclusion rule as the starter feature prep (lane guide section 5):
# impressions_90d > 0, content_age_days >= 90, deduped by content_id.
raw = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
raw = raw.drop_duplicates(subset="content_id")
print(f"Rows after inclusion filter: {len(raw):,}")

FEATURES = ["impressions_90d", "avg_position", "ctr", "days_since_last_update", "word_count"]
ID_COLS = ["content_id", "client_id"]

features = raw[ID_COLS + FEATURES].copy()

# --- Missing value handling ---
# avg_position == 0 / NaN means the page has no ranking to report; treat as "no position" not
# "position zero" (zero would be nonsensical and would badly distort any position-tier split).
features["has_position"] = features["avg_position"] > 0

# word_count missing -> leave NaN visible rather than silently imputing to 0, which would make a
# missing-metadata page look identical to a genuinely thin page.
print(f"\nMissing values per feature:\n{features[FEATURES].isna().sum()}")
print(f"\nRows with no reported position: {(~features['has_position']).sum():,}")

# --- Engineered fields derived only from this same feature window ---
features["position_tier"] = pd.Series(index=features.index, dtype="object")
features.loc[features["has_position"], "position_tier"] = pd.qcut(
    features.loc[features["has_position"], "avg_position"], 5, labels=[1, 2, 3, 4, 5]
)
features["staleness_bucket"] = pd.cut(
    features["days_since_last_update"], bins=[-1, 90, 180, np.inf], labels=["<90d", "90-180d", "180d+"]
)

print(f"\nFinal feature frame shape: {features.shape}")
features.head()

Rows after inclusion filter: 30,000

Missing values per feature:
impressions_90d              0
avg_position                 0
ctr                          0
days_since_last_update       0
word_count                7699
dtype: int64

Rows with no reported position: 1,205

Final feature frame shape: (30000, 10)


,content_id,client_id,impressions_90d,avg_position,ctr,days_since_last_update,word_count,has_position,position_tier,staleness_bucket
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,20,3221.0,True,3,<90d
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,25,2481.0,True,4,<90d
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,20,3515.0,True,5,<90d
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,22,NaN,True,2,<90d
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,14,2803.0,True,5,<90d


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every feature here is an **observed signal**, not a derived product decision (lane guide section 3) — none of them require knowing what FlyRank's app decided to do about the page.

In [9]:
feature_notes = pd.DataFrame([
    {
        "feature": "impressions_90d",
        "meaning": "search impressions in the trailing 90-day window",
        "missing_handling": "none observed post-filter (inclusion rule requires > 0)",
        "type": "numeric, observed signal",
        "available_before_decision_point": True,
    },
    {
        "feature": "avg_position",
        "meaning": "average search ranking position in the window",
        "missing_handling": "0/NaN treated as 'no position' via has_position flag, not imputed to a number",
        "type": "numeric, observed signal",
        "available_before_decision_point": True,
    },
    {
        "feature": "ctr",
        "meaning": "click-through rate in the window",
        "missing_handling": "check for NaN below; flag if any appear alongside nonzero impressions",
        "type": "numeric, observed signal (derived from clicks/impressions upstream)",
        "available_before_decision_point": True,
    },
    {
        "feature": "days_since_last_update",
        "meaning": "days since the content was last edited",
        "missing_handling": "none expected -- content metadata field",
        "type": "numeric, observed signal",
        "available_before_decision_point": True,
    },
    {
        "feature": "word_count",
        "meaning": "content length",
        "missing_handling": "left as NaN rather than imputed to 0 -- see note above",
        "type": "numeric, observed signal",
        "available_before_decision_point": True,
    },
])
print(feature_notes.to_string(index=False))

# All five are available before any decision point because none of them require the future --
# they describe the page's state as of the snapshot, not what happened to it afterward.
n_ctr_missing = features["ctr"].isna().sum()
print(f"\nRows with CTR missing: {n_ctr_missing:,} (inclusion rule already guarantees impressions_90d > 0)")

               feature                                          meaning                                                              missing_handling                                                                type  available_before_decision_point
       impressions_90d search impressions in the trailing 90-day window                       none observed post-filter (inclusion rule requires > 0)                                            numeric, observed signal                             True
          avg_position    average search ranking position in the window 0/NaN treated as 'no position' via has_position flag, not imputed to a number                                            numeric, observed signal                             True
                   ctr                 click-through rate in the window         check for NaN below; flag if any appear alongside nonzero impressions numeric, observed signal (derived from clicks/impressions upstream)                             T

## 3. The leakage hunt

Attacking the five features above directly: could any of them be encoding the future, or encoding a product decision rather than an observed measurement?

In [10]:
# --- Test 1: is any feature actually a disguised product decision flag? ---
banned_substrings = ["health_score", "priority_score", "action_type", "needs_", "is_quick_win", "refresh_tier"]
leaked_cols = [c for c in raw.columns if any(b in c for b in banned_substrings)]
print("Product-flag columns present in raw data:", leaked_cols if leaked_cols else "NONE FOUND -- clean")
print("Product-flag columns used in FEATURES:", [f for f in FEATURES if f in leaked_cols] or "NONE -- clean")

# --- Test 2: does any feature correlate suspiciously well with trend_direction? ---
# trend_direction is the STARTER pipeline's own label (lane guide section 5) -- it must never be
# folded into a feature, since that would let the model (or a rule) copy the answer instead of
# finding real signal.
if "trend_direction" in raw.columns:
    check = raw[FEATURES + ["trend_direction"]].copy()
    print("\ntrend_direction present in raw data -- confirming it is NOT in FEATURES:")
    print("trend_direction in FEATURES:", "trend_direction" in FEATURES)

# --- Test 3: feature window overlap check ---
# All five features are described as trailing/current-state measurements (90d windows, current
# metadata) -- none of them are computed FROM a future window relative to any target this project
# might later define. This needs re-confirming the moment an actual target/label is chosen (w05+),
# since "available now" is only half the leakage question -- the other half is whether the TARGET's
# window overlaps these features' window.
print("\nFeature windows are all trailing/current-state as of the snapshot date.")
print("Re-run this check once a specific target window is defined in a later notebook.")

Product-flag columns present in raw data: NONE FOUND -- clean
Product-flag columns used in FEATURES: NONE -- clean

trend_direction present in raw data -- confirming it is NOT in FEATURES:
trend_direction in FEATURES: False

Feature windows are all trailing/current-state as of the snapshot date.
Re-run this check once a specific target window is defined in a later notebook.


## 4. What I excluded and why

| Excluded | Why |
|---|---|
| `trend_direction` | This is the starter pipeline's own label (lane guide section 5) — using it as a feature would let the rule copy an existing bucket instead of finding independent signal. |
| `health_score`, `priority_score`, `action_type`, refresh flags | Product-computed decision outputs, not shipped in this dataset regardless, and explicitly barred as features by the lane guide (section 4) — feeding a product's own decision back in as an input is a circular result. |
| Raw query/URL/title/client fields | Not present in the release (scrambled before export) and would be a public-safe violation if they were. |
| Other starter-model numeric fields (search volume, competition, CPC, AI sessions, scroll rate, engagement rate) | Not excluded on principle — just outside this five-feature frame's scope. Kept the frame small and tied to Lane 1's specific question rather than importing the full starter feature list wholesale. Worth revisiting if the signal audit shows the five-feature frame is too thin. |
| `content_age_days` | Overlaps conceptually with `days_since_last_update` (both describe content age/freshness) — kept one to avoid two collinear freshness signals in a five-feature frame. |

In [11]:
excluded = [
    "trend_direction", "health_score", "priority_score", "action_type",
    "search_volume", "competition", "cpc", "ai_sessions_90d", "ai_traffic_pct",
    "scroll_rate", "engagement_rate", "content_age_days",
]
present_but_excluded = [c for c in excluded if c in raw.columns]
print("Confirmed present in raw data but excluded from FEATURES:")
print(present_but_excluded)
print("\nConfirmed NOT in FEATURES:", all(c not in FEATURES for c in present_but_excluded))

Confirmed present in raw data but excluded from FEATURES:
['trend_direction', 'search_volume', 'competition', 'cpc', 'ai_sessions_90d', 'ai_traffic_pct', 'scroll_rate', 'engagement_rate', 'content_age_days']

Confirmed NOT in FEATURES: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.